### NumPy: базовый

####Массив numpy Vs список Python

Python - высокоуровневый, интепретируемый и динамический. Списки в нём – списки указателей на объекты (которые могут иметь любой тип), и при выполнении, например, "векторных операций" (с помощью циклов или list comprehensions) интерпретатор каждый раз проверяет, применим ли тип очередного элемента.

То, за счёт чего мы получаем лаконичность кода и высокую скорость разработки, вынуждает наши программы работать медленнее.

Numpy написан частично в питоне, а частично в С, где работа с последовательностями данных организована иначе. Основное понятие и тип данных numpy - это массив (array), в котором все элементы должны быть одного типа, поэтому нет дополнительных вычислительных затрат на проверки. Отсюда прирост в производительности, на некоторых задачах – в сотни раз.

*Если вы используете numpy и если при этом в вашем коде есть циклы, скорее всего, вы делаете что-то не так.*

In [2]:
import numpy as np # импортируем numpy под именем np, это стандартное сокращение

NumPy - это библиотека, созданная для оптимизации математических вычислений, в том числе работы с матрицами. Мы можем реализовать матрицы (и тензоры более высокой мерности) стандартными средствами питона, и мы это делали, но такие вещи все равно будут работать медленно.

Список в питоне может хранить в себе меняющееся число элементов, которые могут быть разного размера: на самом деле он реализован в С как сложная структура, (structure), если кому интересен исходный код в С:

(фигурными скобками обозначается тело определяемого объекта, тип которого - struct (структура). ob_item - это список адресов элементов списка, allocated - количество размещенных в памяти элементов).

    typedef struct {
        PyObject_VAR_HEAD
        PyObject **ob_item;
        Py_ssize_t allocated;
    } PyListObject;
    
То есть, список питона - это связная структура, элементы которой могут валяться в памяти где угодно, а питон их отыскивает с помощью указателей (указатель в С - это переменная, которая содержит адрес объекта, такая переменная записывается со звездочкой: \*var, а массив в С определяется его стартовой позицией в памяти, потому что все элементы должны лежать рядышком и быть одинакового размера, поэтому массив записывается с двумя звездочками).

В компьютерном мире так устроено, что, грубо говоря, пока там питон отыщет все элементы списка в разных уголках памяти, это займет время - а вот когда все элементы лежат рядом, то времени уходит куда меньше. Массивы в numpy реализованы таким образом, что все их элементы одинакового размера и хранятся в памяти вместе, поэтому они и работают гораздо быстрее.

####Типы данных

В связи с реализацией в С у Numpy есть и такая особенность, как немножко свои типы данных (они больше похожи на типы в С). Такие типы есть:

    i - int_ (подтипы int8, int16, int32, int64)
    b - bool_ (bool в numpy является числовым типом, но не подтипом int_)
    u - uint (unsigned integer, подтипы: uint8, uint16, uint32, uint64)
    f - float_ (подтипы: float16, float32, float64, float128)
    c - complex (complex float, подтипы: complex64, complex128)
    m - timedelta (числовой тип, по факту хранится в памяти как int64)
    M - datetime (хранится так же, как timedelta)
    O - object (особый тип, который превращает массив в гибкий контейнер, хранящий указатели, как обычный список Python)
    S или a - bytes_ (байтовая строка ASCII)
    U - str_ (Unicode string)
    V - fixed chunk of memory for other type (void)
    
(void в С ~ NoneType в питоне.)

Типы чисел более дробные: есть беззнаковые инты - у которых не хранится знак - есть инт и флоат разных размеров, 8, 16, 32 или 64 бита: чем больше битов, тем больше чисел может храниться в переменной такого типа.

Эти буковки могут использоваться для задания типа данных ndarray (n-dimensional array) - массиве numpy. Чтобы проверить тип уже созданного массива, используем его атрибут .dtype.

In [3]:
# numpy.ndarray -- это элементы ОДНОГО типа (в numpy их много)

array = [1, 222, 33, 5]
nparray = np.array(array)

print(type(nparray))
print(nparray)
print(nparray.dtype)

array = [1, 222, 33, 5.0]
nparray = np.array(array)

print(nparray)
print(nparray.dtype)

array = [1, 222, 33, "5.0"]
nparray = np.array(array)

print(nparray)
print(nparray.dtype)

<class 'numpy.ndarray'>
[  1 222  33   5]
int64
[  1. 222.  33.   5.]
float64
['1' '222' '33' '5.0']
<U21


А можно и явным образом указать тип с помощью буковки из списка выше. При этом, какие бы мы данные не ввели, numpy их все переведет в нужный нам тип.

In [7]:
arr = np.array([1, 2, 3, 4, 5], dtype=np.str_)
print(arr)

['1' '2' '3' '4' '5']


Либо преобразовать уже имеющийся массив в массив другого типа с помощью метода .astype():

In [8]:
arr1 = np.array([1, 2, 3, 4, 5])
print(arr1)
arr2 = arr1.astype(np.float32)
print(arr2)

[1 2 3 4 5]
[1. 2. 3. 4. 5.]


####Математический смысл массива numpy

Здесь немного сложного:

Центральное понятие для NumPy (а также для PyTorch, например) - это тензор. Это понятие из математики, и тензор на самом деле != матрица, даже многомерная матрица. Тензор - это такой математический объект, который не изменяется при смене системы координат и представлен набором своих компонент и правилом преобразования компонент при смене базиса. Сложно? Если пытаться совсем упростить, то тензор - это всего лишь хранилище данных. Попытаться это понять можно [здесь](https://habr.com/ru/post/261563/). То есть, в тензорах мы храним что? числа, векторы, матрицы. У тензора есть ранг (rank) или измерение: это ровно то, о чем идет речь в NumPy, когда мы указываем измерение ndarray. Внутри тензора могут содержаться, например, n-мерные векторы.

In [11]:
arr = np.array([[1, 2, 3], [4, 5, 6]])
print(arr.ndim)

2


Вот, например, перед нами массив NumPy ранга 2, но при этом векторы, которые в нем содержатся, трехмерные.

Тензор еще считается обобщенной матрицей: матрица - это же табличка чисел $n \cdot m$, а тензор бывает разных рангов. Для маленьких рангов есть отдельные термины:

- 0D - скаляр (число)
- 1D - вектор
- 2D - матрица
- 3D и выше - тензор

В геометрии есть похожее соотношение: квадрат - куб - симплекс (любая фигура со сторонами одинаковой длины - 1 - в n-мерном пространстве называется n-мерный симплекс)

Итак, как задать массив в NumPy и проверить его мерность, можно понять из кода выше. Можно мерность задать при создании массива, при этом передавать ему список списков списков необязательно:

In [12]:
arr = np.array([1, 2, 3, 4, 5], ndmin=5)
print(arr)

[[[[[1 2 3 4 5]]]]]


В таком случае numpy начнет вкладывать наши списки списков начиная с самого внутреннего измерения.

####Векторизация

Векторизация — это отказ от циклов на уровне Python в пользу одновременной обработки всех элементов массива на уровне быстрого скомпилированного C-кода.

В NumPy мы просто пишем математическое выражение для всего массива сразу:

In [10]:
prices_np = np.array([100, 200, 300])
new_prices_np = prices_np + 10  # Векторизованная операция!

print(new_prices_np)

[110 210 310]


Почему это работает мгновенно?

NumPy заранее знает, что внутри массива лежат, например, только int64. Ему не нужно проверять тип каждого элемента. Он передает этот непрерывный блок памяти процессору, и тот одной командой (на уровне архитектуры SIMD — Single Instruction, Multiple Data) прибавляет десятку сразу к целой пачке чисел.

Четыре основных сценария векторизации в numpy:

- Массив и скаляр (число): операция применяется
к каждому элементу массива.
- Массив и массив (одинаковой длины): операция
производится поэлементно (по индексам).
- Векторизованные математические функции
(np.sin, np.log, np.exp): модуль math из
Python не умеет работать с массивами. В NumPy
для этого есть свои встроенные функции.
- Векторизованные сравнения: результат - массив
из True и False.

In [31]:
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])
print(a + b)
print(a < b)

[5 7 9]
[ True  True  True]


####Булевы маски

Булева маска — это массив, который создается
на основе другого массива. Он состоит только
из элементов типа bool, соответствующих
элементам исходного массива и отражающих то,
подходят ли эти исходные элементы под
некоторое заданное условие.
- Булева маска всегда имеет точно такой же
размер, как исходный массив.
- Булеву маску можно использовать для
фильтрации данных, если передать ее внутрь
квадратных скобок исходного массива.

In [32]:
balances = np.array([1500, -200, 0, 450, -15])
debtor_mask = balances < 0
print(debtor_mask)
#Фильтруем данные по маске
bad_balances = balances[debtor_mask]
print(bad_balances)

[False  True False False  True]
[-200  -15]


У вас может возникнуть желание объединить две (или больше) булевы маски с помощью логической операции. Но тут есть одна тонкость: обычные логические операторы Python не умеют работать с массивами. Поэтому в numpy есть свои логические операторы, которые применяются к булевым маскам поэлементно и похожи на операторы из С: вместо and пишем &, вместо or пишем | и вместо not пишем ~

In [34]:
debtor_mask1 = balances > -100
not_so_bad_balances = balances[debtor_mask & debtor_mask1]
print(not_so_bad_balances)

[-15]


####Агрегации и axis

Агрегация в NumPy — это процесс преобразования массива чисел в одно общее значение (или массив меньшего размера).
Основные функции агрегации
• np.sum() — считает сумму всех элементов.
• np.mean() — вычисляет среднее арифметическое.
• np.min() / np.max() — находит минимальный или максимальный элемент.
• np.median() — определяет медиану (срединное значение).
• np.std() — считает стандартное отклонение.
• np.var() — вычисляет дисперсию.
• np.prod() — считает произведение всех элементов.

Как это работает?

По умолчанию любая функция агрегации схлопывает весь массив в один скаляр (одно число):




In [13]:
arr = np.array([1, 2, 3, 4])
print(np.sum(arr))

10


Для многомерных массивов можно применять агрегацию по измерениям и получать на выходе новый массив.

В NumPy (как потом увидим, и в Pandas) измерение массива еще называются осью (axis). Если выставить для агрегации двумерного массива параметр axis=0, операция выполняется по столбцам (сверху вниз, «схлопывает» строки);  — операция выполняется по строкам (слева направо, «схлопывает» столбцы).

In [14]:
matrix = np.array([[1, 2], [3, 4]])

print(np.sum(matrix, axis=0))  # Сумма по столбцам: 1+3 и 2+4
print(np.sum(matrix, axis=1))  # Сумма по строкам: 1+2 и 3+4

[4 6]
[3 7]


При агренгации по осям параметр keepdims=True позволяет
сохранить исходную размерность
массива, не превращая его в
плоский вектор, что критически
важно для корректного вычитания
или деления матриц.

In [15]:
matrix = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
print("Общая сумма:", np.sum(matrix)) # Результат: 45
print("Сумма по столбцам:", np.sum(matrix, axis=0))
print("Максимум по строкам:", np.max(matrix, axis=1))
res_flat = np.sum(matrix, axis=1)
print("Без keepdims:", res_flat, "Форма:", res_flat.shape)
res_dims = np.sum(matrix, axis=1, keepdims=True)
print("С keepdims:\n", res_dims, "Форма:", res_dims.shape)

Общая сумма: 45
Сумма по столбцам: [12 15 18]
Максимум по строкам: [3 6 9]
Без keepdims: [ 6 15 24] Форма: (3,)
С keepdims:
 [[ 6]
 [15]
 [24]] Форма: (3, 1)


Есть также специальные агрегации.

Обычные функции агрегации вернут nan, если в
массиве есть хотя бы одно неопределенное
значение. Для корректного подсчета по
оставшимся числам используются специальные
nan-версии функций:

- nansum для безопасной суммы;
- nanmean для безопасного среднего;
- nanstd для стандартного отклонения, и т.д.

Логические агрегации спользуются для проверки
условий на массивах с булевыми значениями:

- np.any — хотя бы один элемент равен True?
- np.all — все элементы равны True?

In [16]:
bool_arr = np.array([True, False, True])
print("Any:", np.any(bool_arr))
print("All:", np.all(bool_arr))

Any: True
All: False


#### Индексирование

Как можно обращаться к элементам массива? По индексам. Только они устроены немножко похитрее, чем стандартные питоньи (где нам пришлось  бы писать [0][0][0] для первого элемента первой строки первого... эээ, даже не знаю. Ранга. :)

In [17]:
arr = np.array([[[1, 2], [3, 4]], [[5, 6], [7, 8]]])
print(arr)

[[[1 2]
  [3 4]]

 [[5 6]
  [7 8]]]


Вот у нас трехмерный массив. Допустим, мы хотим обратиться к четверке. Четверка находится в первой из двух матриц, во второй строке первой матрицы, на второй позиции этой строки.

In [18]:
print(arr[0, 1, 1])

4


Супер. Также можно использовать срезы: например, можно получить всю вторую строку первой матрицы.

In [19]:
print(arr[0, 1, :])

[3 4]


И даже весь второй столбец.

In [20]:
print(arr[0, :, 1])

[2 4]


#### Форма

Форму матрицы можно изменять с помощью метода reshape.

In [21]:
arr = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9])
print(arr.reshape(3, 3))

[[1 2 3]
 [4 5 6]
 [7 8 9]]


То есть, что передается в reshape? Столько цифр, сколько хотим измерений (я передала 2 - значит, я хочу матрицу), каждая цифра говорит о том, сколько я хочу элементов в каждом из этих измерений (три строки, три столбца). Tricky part здесь в том, что общее число элементов должно делиться на эти числа: то есть, все переданные числа, перемноженные друг на друга, должны давать число всех элементов ($ 3 \cdot 3 = 9$).

Если мы не можем так с ходу вычислить, сколько нам нужно элементов, можно попробовать один из них передать как минус единицу, тогда питон попробует вычислить его сам:

In [22]:
arr = np.array(range(1, 13))
print(arr.reshape(3, 2, -1))

[[[ 1  2]
  [ 3  4]]

 [[ 5  6]
  [ 7  8]]

 [[ 9 10]
  [11 12]]]


-1 могло бы стоять вместо любой из цифр. Увы, если у вас не хватает элементов в массиве, чтобы разложить его на нужное количество, остается только добавлять эти элементы до того, как вы будете решейпить.

In [24]:
arr = np.array(range(1, 12))
arr = np.append(arr, 0)  # можно было бы и None добавить, но лучше уж 0, чтобы numpy не пришлось менять тип данных на object
print(arr)

[ 1  2  3  4  5  6  7  8  9 10 11  0]


Превратить любой массив в одномерный можно разными способами, но в частности, можно просто отрешейпить в -1:

In [26]:
curled = np.array([[1, 2, 3], [4, 5, 6]])
flat = curled.reshape(-1)
print(flat)

[1 2 3 4 5 6]


#### view vs base

Также еще одна важная особенность массивов NumPy, о которой следует знать - это различие между base и view object. Для чего эта сложность? Это мы с вами на учебных занятиях возимся с массивчиками, в которых по три с половиной элемента, а в реальных программах массивы могут содержать миллиарды чисел. Случайно копировать такой массив, конечно, не улыбается, поэтому все библиотеки типа NumPy стараются избегать копирования, как только могут (ванильный питон тоже это делает - вспомните особенности копирования списков и других изменяемых объектов).

view-массив - это такой массив, который собственных данных не содержит, а просто берет их из какого-то base-массива, но отображает по-своему.

reshape как раз всегда возвращает такие вот "виды".

Чтобы проверить, вид перед вами или база, можно обратиться к атрибуту .base:

In [40]:
arr_view = arr.reshape(3, 4)
print(arr_view)
print(arr_view.base)

[[ 1  2  3  4]
 [ 5  6  7  8]
 [ 9 10 11  0]]
[ 1  2  3  4  5  6  7  8  9 10 11  0]


Если массив оригинальный и владеет своими данными, то base вернет пустоту, в ином случае - хозяина данных. То есть, как легко догадаться, если мы будем изменять данные в массиве arr, данные в массиве arr_view тоже изменятся:

In [43]:
arr[2] = 123
print(arr_view)

[[  1   2 123   4]
 [  5   6   7   8]
 [  9  10  11   0]]


Явным образом создать копию можно с помощью метода copy().

#### Поиск по значению и сортировка

Можно искать по массиву элемент, который чему-нибудь равен:

In [ ]:
arr = np.array([[1, 2, 3], [2, 3, 1]])
print(np.where(arr==3))

(array([0, 1]), array([2, 1]))


В этом случае вернутся индексы найденных элементов.

where умеет еще и заменять найденное. При этом он заменяет элемент в исходном массиве, а не создает view:

In [50]:
new_arr = np.where(arr == 3, 333, arr)
print(new_arr)
print(new_arr.base)

[[  1   2 333]
 [  2 333   1]]
None


Сортировать массивы можно с помощью np.sort(), которая возвращает копию массива, а оригинальный массив оставляет нетронутым.

#### Итерирование

Как итерироваться по массивам? Можно тупо по индексам с помощью for, но для многомерных массивов это может быть сложновато, поэтому есть специальная функция:

In [ ]:
for elem in np.nditer(arr):
    print(elem, end='\t')

1	2	123	4	5	6	7	8	9	10	11	0	

Есть и альтернатива enumerate для массивов:

In [ ]:
for i, elem in np.ndenumerate(arr):
    print(i, elem, end='\t')

(0,) 1	(1,) 2	(2,) 123	(3,) 4	(4,) 5	(5,) 6	(6,) 7	(7,) 8	(8,) 9	(9,) 10	(10,) 11	(11,) 0	

#### Конкатенация и разделение

Окей, а если мы хотим объединить два массива, при этом не складывая их? Тут есть несколько вариантов:

In [44]:
arr1 = np.array([[1, 2, 3]])
arr2 = np.array([[4, 5, 6]])
arr = np.concatenate((arr1, arr2))
arrv = np.concatenate((arr1, arr2), axis=1)
print(arr)
print(arrv)


[[1 2 3]
 [4 5 6]]
[[1 2 3 4 5 6]]


In [45]:
print(np.stack((arr1, arr2)))

[[[1 2 3]]

 [[4 5 6]]]


Тут немного неочевидно - но stack возвращает массив большей мерности, когда склеивает, а concatenate - нет.

Посплитить тоже можно:

In [46]:
print(np.array_split(arr, 2))

[array([[1, 2, 3]]), array([[4, 5, 6]])]
